In [2]:
# X_{n+1} = e^{−1} X_n+1−e^{−2} ξ_n
import numpy as np
import pandas as pd


def simulate_ou(n_steps, gamma, dt, random_state, x0=0.0):
    """AR(1) sampled from the exact OU transition at physical step size dt."""
    rng = np.random.default_rng(random_state)
    a = np.exp(-gamma * dt)
    b = np.sqrt(1.0 - a**2)
    x = np.empty(n_steps, dtype=float)
    x[0] = float(x0)
    for t in range(n_steps - 1):
        x[t + 1] = a * x[t] + b * rng.standard_normal()
    return pd.DataFrame({"x": x})


In [2]:
# General gaussian invariant distribution: ξ_n ∼ N(0,1) for OU, not actually used for this figure
# Actual density used is below
def stationary_density(x, theta=1.0, mu=0.0, sigma=1.0):
    var = sigma**2 / (2 * theta)
    return np.exp(-((x - mu) ** 2) / (2 * var)) / np.sqrt(2 * np.pi * var)


In [3]:
# OU specific normalisation constant

from kooplearn._linalg import weighted_norm


def ou_normalisation(functions, x, density):
    #    abs2 = np.abs(functions) ** 2
    #    norms = np.sqrt(np.trapezoid(abs2 * density[:, None], x[:, 0], axis=0))
    W = np.diag(density.reshape(-1))
    norms = weighted_norm(functions, W)
    norms = np.maximum(norms, 1e-12)
    functions = functions / norms
    return functions


In [4]:
# Some reused variables
from kooplearn.preprocessing import FeatureFlattener

gamma = 1e-4 # same as paper

x = np.linspace(-4, 4, 1025)[:, None]

flattener = FeatureFlattener()
x_flat = flattener.fit_transform(x)

# This is the exact density used in the paper for OU
density = np.exp(-0.5 * x_flat**2) / np.sqrt(2 * np.pi)


In [ ]:
# First, Fig. 2 but with OU

from collections import defaultdict

import matplotlib.pyplot as plt
from tqdm import tqdm

from kooplearn.kernel import KernelRidge


def fit_and_estimate(reduced_rank, x, density, random_state):
    # Substitute Langevin with OU

    data = simulate_ou(n_steps=n_steps, gamma=gamma, dt=dt, random_state=0, x0=0.0)
    data = data.iloc[::subsample][:n_train]

    # Model definition, same as fig 2 for now
    model = KernelRidge(
        n_components=5,
        reduced_rank=reduced_rank,
        gamma=12.5,
        kernel="rbf",
        alpha=1e-6,
        random_state=random_state,
    )

    # Fit and estimate eigenfunctions
    model.fit(data)  # fit transfer op model
    values, functions = model.eig(
        eval_right_on=x
    )  # (right) eigenvalue estimation, evaluate on array x
    sort_perm = np.flip(np.argsort(np.abs(values)))  # Order decreasingly
    values, functions = values[sort_perm], functions[:, sort_perm]
    functions = ou_normalisation(functions, x, density)
    return functions


# Run functions for both RRR (reduced rank) and kDMD (full rank) estimators
dt = 1e-4
n_repetitions = 10
results = defaultdict(list)
for method, reduced_rank in zip(["Principal Components (kDMD)", "Reduced Rank"], [False, True]):
    for i in tqdm(range(n_repetitions)):
        results[method].append(fit_and_estimate(reduced_rank, x, density, i))

# Print results
fig, axs = plt.subplots(ncols=4, figsize=(9, 2))
for fun_id, ax in enumerate(axs):
    for method, functions in results.items():
        color = "tab:blue" if "Principal" in method else "tab:orange"
        ax.plot(x, functions[0][:, fun_id], color=color, label=method)
    ax.set_title(rf"Eigenfunction $\psi_{fun_id}$", fontsize=9)
    ax.set_xlim(-1, 1)
handles, labels = ax.get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    ncols=2,
    frameon=False,
    bbox_to_anchor=(0.0, 1.05, 1.0, 0.102),
)
plt.tight_layout()
plt.show()


## Figure 1 reproduction
### Kernel family
$$k_{\Pi,\nu}(x,x')=\sum_{i\in\mathbb N}\mu_{\Pi(i)}^{2\nu} f_i(x)f_i(x')$$

#### Trial run to test plotting

In [ ]:
# Spectral rates paper fig. 1-style eigenfunction comparison:
# - 3 columns = good / bad / ugly kernels
# - overlay PCR (blue) and RRR (orange)
# - show mean estimated eigenfunction across trials for the first few modes

from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from tqdm import tqdm

from kooplearn.kernel import KernelRidge


# ----------------------------------------------------------------------------------------------
# Hermite helpers / kernel family - in line with Kostic et al. Kernel construction in Example 3
# ----------------------------------------------------------------------------------------------
def hermite_features(x, M):
    """Normalized probabilists' Hermite functions f_n = He_n/sqrt(n!),
    via sqrt(n) f_n = x f_{n-1} - sqrt(n-1) f_{n-2}. No factorial, no hermeval."""
    x = np.asarray(x).reshape(-1)
    f = np.empty((x.shape[0], M))
    f[:, 0] = 1.0
    if M > 1:
        f[:, 1] = x
    for n in range(2, M):
        f[:, n] = (x * f[:, n - 1] - np.sqrt(n - 1) * f[:, n - 2]) / np.sqrt(n)
    return f


def hermite_kernel(kind, r, M):
    w = kernel_weights(kind, r, M)
    cache = {}

    def get_features(x):
        key = np.asarray(x).ravel().tobytes()
        feats = cache.get(key)
        if feats is None:
            feats = hermite_features(x, M)[0]
            cache[key] = feats
        return feats

    def kernel(x, y):
        fx, fy = get_features(x), get_features(y)
        return float(np.sum(w * fx * fy))

    return kernel


def kernel_permutation(kind, r, M):
    pi = np.arange(M)

    if kind == "good":
        return pi

    if kind == "bad":
        if 2 * r > M:
            raise ValueError(f"Need M >= 2*r for bad kernel, got M={M}, r={r}")
        pi2 = pi.copy()
        a = np.arange(r)
        b = np.arange(r, 2 * r)
        pi2[a] = b[::-1]
        pi2[b] = a[::-1]
        return pi2

    if kind == "ugly":
        return pi[::-1]

    raise ValueError(kind)


def kernel_weights(kind, r, M):
    mu = np.exp(-np.arange(M))
    nu = 1.0 if kind == "good" else (1.0 / (r**2) if kind == "bad" else r**2)
    pi = kernel_permutation(kind, r, M)
    return mu[pi] ** (2.0 * nu)


# -----------------------------
# OU reference eigenfunctions
# -----------------------------
def compute_ou_eig(gamma, lag, num_components):
    n = np.arange(num_components)
    return np.exp(-n * gamma * lag)


# -----------------------------
# Sign-align estimated functions to a reference
# -----------------------------
def standardize_sign(eigenfunction, reference, x):
    eigenfunction_cut = cut_functions_to_domain(eigenfunction, x)
    reference_cut = cut_functions_to_domain(reference, x)
    norm_p = np.linalg.norm(eigenfunction_cut + reference_cut)
    norm_m = np.linalg.norm(eigenfunction_cut - reference_cut)
    if norm_p <= norm_m:
        return -1.0 * eigenfunction
    else:
        return eigenfunction


def cut_functions_to_domain(functions, x, x_lims=(-1, 1)):
    mask = (x >= x_lims[0]) & (x <= x_lims[1])
    return functions[mask]


# -----------------------------
# Parameters
# -----------------------------
kernels = ["good", "bad", "ugly"]
methods = [("PCR", False), ("RRR", True)]

n_train = 1000
subsample = 50
n_steps = n_train * subsample
n_trials = 5  # increase toward paper settings for smoother curves
n_show = 3  # number of leading eigenfunctions to plot
M = 10  # kernel truncation level
dt = 1e-4
lag = dt * subsample * 5
n_components = 5
r = n_show
true_eigs = compute_ou_eig(gamma, lag, n_components)

x_plot = x.reshape(-1)

results = defaultdict(list)


# -----------------------------
# Fit model and return eigenvalues
# -----------------------------
def fit_and_estimate_values(reduced_rank, x, density, random_state, kind, M, n_show):
    out = np.full(n_show, np.nan, dtype=float)

    try:
        data = simulate_ou(
            n_steps=n_steps,
            gamma=gamma,
            dt=dt,
            random_state=random_state,
            x0=0.0,
        )
        data = data.iloc[::subsample]
        data = data[:n_train]

        if random_state < 2:
            tqdm.write(
                f"{kind}-{reduced_rank}-trial {random_state}: {data.head().to_numpy().ravel()[:5]}"
            )

        model = KernelRidge(
            n_components=n_show,
            reduced_rank=reduced_rank,
            kernel=hermite_kernel(kind, r, M),
            alpha=0,
            random_state=random_state,
        )

        model.fit(data)
        values, functions = model.eig(eval_right_on=x)

        values = np.asarray(values)
        values = np.real_if_close(values)
        values = np.real(values)

        if values.ndim == 0:
            values = np.array([float(values)])

        sort_perm = np.flip(np.argsort(np.abs(values)))
        values = values[sort_perm]

        k = min(n_show, len(values))
        out[:k] = values[:k]

    except Exception as e:
        tqdm.write(f"trial {random_state} failed: {e}")

    return out


# -----------------------------
# Collect spectra
# -----------------------------
results = defaultdict(list)

for kind in kernels:
    for method, reduced_rank in methods:
        for trial in tqdm(range(n_trials), desc=f"{kind}-{method}"):
            vals = fit_and_estimate_values(
                reduced_rank=reduced_rank,
                x=x,
                density=density,
                random_state=trial,
                kind=kind,
                M=M,
                n_show=n_show,
            )
            results[(kind, method)].append(vals)

for key in results:
    results[key] = np.asarray(results[key], dtype=float)
    print(key, results[key].shape)

# -----------------------------
# Plot
# -----------------------------
# -----------------------------
# Shared x-range per kernel
# -----------------------------
kernel_ranges = {}
for kind in kernels:
    combined = []
    for method, _ in methods:
        arr = results[(kind, method)]
        combined.append(arr[np.isfinite(arr)])
    combined = np.concatenate(combined) if len(combined) else np.array([])

    if combined.size == 0:
        kernel_ranges[kind] = (0.0, 1.0)
        continue

    if kind in ("good", "bad"):
        xlo = min(np.min(true_eigs) - 0.01, np.nanmin(combined) - 0.005)
        xhi = max(np.max(true_eigs) + 0.005, np.nanmax(combined) + 0.005)
    else:
        span = np.nanmax(combined) - np.nanmin(combined)
        pad = max(0.03, 0.08 * span if span > 0 else 0.05)
        xlo = np.nanmin(combined) - pad
        xhi = np.nanmax(combined) + pad

    kernel_ranges[kind] = (xlo, xhi)

plt.style.use("default")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "0.2",
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
})

fig, axs = plt.subplots(
    nrows=2,
    ncols=3,
    figsize=(12.8, 5.4),
    sharex=False,
    sharey=False,
)

mode_colors = ["#4C78A8", "#F58518", "#54A24B"]

for row, (method, _) in enumerate(methods):
    for col, kind in enumerate(kernels):
        ax = axs[row, col]
        vals = np.asarray(results[(kind, method)], dtype=float)

        xlo, xhi = kernel_ranges[kind]
        n_bins = 70 if kind in ("good", "bad") else 55
        bins = np.linspace(xlo, xhi, n_bins)

        # mode-wise empirical distributions
        for m in range(min(n_show, vals.shape[1])):
            vm = vals[:, m]
            vm = vm[np.isfinite(vm)]
            if vm.size == 0:
                continue

            ax.hist(
                vm,
                bins=bins,
                density=True,
                histtype="stepfilled",
                alpha=0.18,
                lw=0,
                color=mode_colors[m % len(mode_colors)],
            )
            ax.hist(
                vm,
                bins=bins,
                density=True,
                histtype="step",
                lw=1.4,
                color=mode_colors[m % len(mode_colors)],
                label=f"Mode {m + 1}" if (row == 0 and col == 0) else None,
            )

        # true Koopman eigenvalue lines
        for j, ev in enumerate(true_eigs[:n_show]):
            if xlo <= ev <= xhi:
                ax.axvline(
                    float(ev),
                    color="black",
                    ls=(0, (4, 2)),
                    lw=0.9,
                    alpha=0.85,
                    label="Koopman eigenvalue" if (row == 0 and col == 0 and j == 0) else None,
                )

        ax.set_title(f"{method} / {kind}", pad=6)
        ax.set_xlim(xlo, xhi)
        ax.grid(True, axis="y", alpha=0.18, lw=0.6)
        ax.grid(False, axis="x")

        if col == 0:
            ax.set_ylabel("Density")
        if row == len(methods) - 1:
            ax.set_xlabel("Estimated eigenvalue")

        # cleaner frame
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

legend_handles = [
    Line2D([0], [0], color=mode_colors[0], lw=1.6, label="Mode 1"),
    Line2D([0], [0], color=mode_colors[1], lw=1.6, label="Mode 2"),
    Line2D([0], [0], color=mode_colors[2], lw=1.6, label="Mode 3"),
    Line2D([0], [0], color="black", lw=0.9, ls=(0, (4, 2)), label="Koopman eigenvalue"),
]

fig.legend(
    handles=legend_handles,
    loc="upper center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, 1.01),
    handlelength=2.2,
    columnspacing=1.6,
)

fig.subplots_adjust(top=0.84, wspace=0.25, hspace=0.32)
plt.show()

#### Inspecting returned numbers to determine best plotting approach

In [ ]:
print(kind, method, "min/max per mode:")
print(np.nanmin(vals, axis=0), np.nanmax(vals, axis=0))
print("finite counts:", np.sum(np.isfinite(vals), axis=0))


for key, arr in results.items():
    print(key)
    print(arr)


#### Alternative plotting 

In [ ]:
# -----------------------------
# Plot
# -----------------------------

# Compute one x-range per kernel, shared across PCR/RRR rows in that column
kernel_ranges = {}
for kind in kernels:
    combined = []
    for method, _ in methods:
        arr = np.asarray(results[(kind, method)], dtype=float)
        combined.append(arr[np.isfinite(arr)])
    combined = np.concatenate(combined)

    if combined.size == 0:
        kernel_ranges[kind] = (0.0, 1.0)
        continue

    if kind in ("good", "bad"):
        xlo = min(0.965, np.nanmin(combined) - 0.002)
        xhi = max(1.002, np.nanmax(combined) + 0.002)
    else:
        span = np.nanmax(combined) - np.nanmin(combined)
        pad = max(0.03, 0.08 * span if span > 0 else 0.05)
        xlo = np.nanmin(combined) - pad
        xhi = np.nanmax(combined) + pad

    kernel_ranges[kind] = (xlo, xhi)

fig, axs = plt.subplots(
    nrows=2,
    ncols=3,
    figsize=(13.5, 5.8),
    sharex=False,
    sharey=False,
)

mode_colors = ["tab:blue", "tab:orange", "tab:green"]

for row, (method, _) in enumerate(methods):
    for col, kind in enumerate(kernels):
        ax = axs[row, col]
        vals = np.asarray(results[(kind, method)], dtype=float)  # shape (n_trials, n_show)

        xlo, xhi = kernel_ranges[kind]
        bins = np.linspace(xlo, xhi, 60)

        # plot mode-wise empirical distributions
        for m in range(min(n_show, vals.shape[1])):
            vm = vals[:, m]
            vm = vm[np.isfinite(vm)]
            if vm.size == 0:
                continue

            ax.hist(
                vm,
                bins=bins,
                density=True,
                histtype="step",
                lw=1.8,
                color=mode_colors[m % len(mode_colors)],
                label=f"Mode {m + 1}" if (row == 0 and col == 0) else None,
            )

        # true Koopman eigenvalue lines, only if visible in this panel
        for j, ev in enumerate(true_eigs[:n_show]):
            if xlo <= ev <= xhi:
                ax.axvline(
                    float(ev),
                    color="black",
                    ls="--",
                    lw=1.0,
                    alpha=0.8,
                    label="True Koopman eigenvalue" if (row == 0 and col == 0 and j == 0) else None,
                )

        ax.set_title(f"{method} / {kind}", fontsize=12)
        ax.set_xlim(xlo, xhi)
        ax.grid(alpha=0.20)

        if col == 0:
            ax.set_ylabel("Density")
        if row == len(methods) - 1:
            ax.set_xlabel("Estimated eigenvalue")

legend_handles = [
    Line2D([0], [0], color="tab:blue", lw=1.8, label="Mode 1"),
    Line2D([0], [0], color="tab:orange", lw=1.8, label="Mode 2"),
    Line2D([0], [0], color="tab:green", lw=1.8, label="Mode 3"),
    Line2D([0], [0], color="black", lw=1.0, ls="--", label="True Koopman eigenvalue"),
]

fig.legend(
    handles=legend_handles,
    loc="upper center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, 1.03),
)

fig.suptitle(
    "OU spectrum under good, bad, and ugly kernels",
    fontsize=14,
    y=1.08,
)

plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------
# Fit model and return eigenvalues
# -----------------------------
alpha = 1e-10

def fit_and_estimate_values(reduced_rank, x, density, random_state, kind, M, n_show):
    out = np.full(n_show, np.nan, dtype=float)
    method_name = "RRR" if reduced_rank else "PCR"
    local_n_show = 2 if (kind == "ugly" and method_name == "RRR") else n_show

    try:
        data = simulate_ou(
            n_steps=n_steps,
            gamma=gamma,
            dt=dt,
            random_state=random_state,
            x0=0.0,
        )
        data = data.iloc[::subsample]
        data = data[:n_train]

        model = KernelRidge(
            n_components=local_n_show,
            reduced_rank=reduced_rank,
            kernel=hermite_kernel(kind, r, M),
            alpha=alpha,
            random_state=random_state,
        )

        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model.fit(data)
            values, functions = model.eig(eval_right_on=x)

        for w in caught:
            tqdm.write(
                f"WARNING [{kind}/{method_name}/trial {random_state}] "
                f"{w.category.__name__}: {w.message}"
            )

        values = np.asarray(values)
        values = np.real_if_close(values)
        values = np.real(values)

        if values.ndim == 0:
            values = np.array([float(values)])

        sort_perm = np.flip(np.argsort(np.abs(values)))
        values = values[sort_perm]

        k = min(n_show, len(values))
        out[:k] = values[:k]

    except Exception as e:
        tqdm.write(f"FAILED [{kind}/{method_name}/trial {random_state}] {e}")

    return out


for kind in kernels:
    for method, reduced_rank in methods:
        for trial in tqdm(range(n_trials), desc=f"{kind}-{method}"):
            vals = fit_and_estimate_values(
                reduced_rank=reduced_rank,
                x=x,
                density=density,
                random_state=trial,
                kind=kind,
                M=M,
                n_show=n_show,
            )


#### Combined fixes

In [ ]:
import warnings
from collections import Counter

# -----------------------------
# Parameters
# -----------------------------
kernels = ["good", "bad", "ugly"]
methods = [("PCR", False), ("RRR", True)]

n_train = 2000
subsample = 100
n_steps = n_train * subsample
n_trials = 20  # increase = smoother curves
n_show = 3  # number of leading eigenfunctions to plot
M = 10  # kernel truncation level
dt = 1e-4
lag = dt * subsample * 2
n_components = 5
r = n_show
true_eigs = compute_ou_eig(gamma, lag, n_components)

x_plot = x.reshape(-1)

results = defaultdict(list)


# -----------------------------
# Fit model and return eigenvalues
# -----------------------------


warning_log = []


alpha = 1e-11


def fit_and_estimate_values(reduced_rank, x, density, random_state, kind, M, n_show):
    out = np.full(n_show, np.nan, dtype=float)
    method_name = "RRR" if reduced_rank else "PCR"
    local_n_show = 2 if (kind == "ugly" and method_name == "RRR") else n_show

    try:
        data = simulate_ou(
            n_steps=n_steps,
            gamma=gamma,
            dt=dt,
            random_state=random_state,
            x0=0.0,
        )
        data = data.iloc[::subsample]
        data = data[:n_train]

        model = KernelRidge(
            n_components=local_n_show,
            reduced_rank=reduced_rank,
            kernel=hermite_kernel(kind, r, M),
            alpha=alpha,
            random_state=random_state,
        )

        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model.fit(data)
            values, functions = model.eig(eval_right_on=x)

        for w in caught:
            tqdm.write(
                f"WARNING [{kind}/{method_name}/trial {random_state}] "
                f"{w.category.__name__}: {w.message}"
            )

        values = np.asarray(values)
        values = np.real_if_close(values)
        values = np.real(values)

        if values.ndim == 0:
            values = np.array([float(values)])

        sort_perm = np.flip(np.argsort(np.abs(values)))
        values = values[sort_perm]

        k = min(n_show, len(values))
        out[:k] = values[:k]

    except Exception as e:
        tqdm.write(f"FAILED [{kind}/{method_name}/trial {random_state}] {e}")

    return out


# -----------------------------
# Collect spectra
# -----------------------------
results = defaultdict(list)


for kind in kernels:
    for method, reduced_rank in methods:
        for trial in tqdm(range(n_trials), desc=f"{kind}-{method}"):
            vals = fit_and_estimate_values(
                reduced_rank=reduced_rank,
                x=x,
                density=density,
                random_state=trial,
                kind=kind,
                M=M,
                n_show=n_show,
            )
            results[(kind, method)].append(vals)

for key in results:
    results[key] = np.asarray(results[key], dtype=float)
    print(key, results[key].shape)

summary = Counter((w["kind"], w["method"], w["warning_type"], w["message"]) for w in warning_log)

for key, count in summary.items():
    print(count, key)

# -----------------------------
# Plot
# -----------------------------

# Compute one x-range per kernel, shared across PCR/RRR rows in that column
kernel_ranges = {}
for kind in kernels:
    combined = []
    for method, _ in methods:
        arr = np.asarray(results[(kind, method)], dtype=float)
        combined.append(arr[np.isfinite(arr)])
    combined = np.concatenate(combined)

    if combined.size == 0:
        kernel_ranges[kind] = (0.0, 1.0)
        continue

    if kind in ("good", "bad"):
        xlo = min(0.965, np.nanmin(combined) - 0.002)
        xhi = max(1.002, np.nanmax(combined) + 0.002)
    else:
        span = np.nanmax(combined) - np.nanmin(combined)
        pad = max(0.03, 0.08 * span if span > 0 else 0.05)
        xlo = np.nanmin(combined) - pad
        xhi = np.nanmax(combined) + pad

    kernel_ranges[kind] = (xlo, xhi)

fig, axs = plt.subplots(
    nrows=2,
    ncols=3,
    figsize=(13.5, 5.8),
    sharex=False,
    sharey=False,
)

mode_colors = ["tab:blue", "tab:orange", "tab:green"]

for row, (method, _) in enumerate(methods):
    for col, kind in enumerate(kernels):
        ax = axs[row, col]
        vals = np.asarray(results[(kind, method)], dtype=float)  # shape (n_trials, n_show)

        xlo, xhi = kernel_ranges[kind]
        bins = np.linspace(xlo, xhi, 60)

        # plot mode-wise empirical distributions
        for m in range(min(n_show, vals.shape[1])):
            vm = vals[:, m]
            vm = vm[np.isfinite(vm)]
            if vm.size == 0:
                continue

            ax.hist(
                vm,
                bins=bins,
                density=True,
                histtype="step",
                lw=1.8,
                color=mode_colors[m % len(mode_colors)],
                label=f"Mode {m + 1}" if (row == 0 and col == 0) else None,
            )

        # true Koopman eigenvalue lines, only if visible in this panel
        for j, ev in enumerate(true_eigs[:n_show]):
            if xlo <= ev <= xhi:
                ax.axvline(
                    float(ev),
                    color="black",
                    ls="--",
                    lw=1.0,
                    alpha=0.8,
                    label="True Koopman eigenvalue" if (row == 0 and col == 0 and j == 0) else None,
                )

        ax.set_title(f"{method} / {kind}", fontsize=12)
        ax.set_xlim(xlo, xhi)
        ax.grid(alpha=0.20)

        if col == 0:
            ax.set_ylabel("Density")
        if row == len(methods) - 1:
            ax.set_xlabel("Estimated eigenvalue")

legend_handles = [
    Line2D([0], [0], color="tab:blue", lw=1.8, label="Mode 1"),
    Line2D([0], [0], color="tab:orange", lw=1.8, label="Mode 2"),
    Line2D([0], [0], color="tab:green", lw=1.8, label="Mode 3"),
    Line2D([0], [0], color="black", lw=1.0, ls="--", label="True Koopman eigenvalue"),
]

fig.legend(
    handles=legend_handles,
    loc="upper center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, 1.03),
)

# -----------------------------
# Plot
# -----------------------------

# Compute one x-range per kernel, shared across PCR/RRR rows in that column
kernel_ranges = {}
for kind in kernels:
    combined = []
    for method, _ in methods:
        arr = np.asarray(results[(kind, method)], dtype=float)
        combined.append(arr[np.isfinite(arr)])
    combined = np.concatenate(combined)

    if combined.size == 0:
        kernel_ranges[kind] = (0.0, 1.0)
        continue

    if kind in ("good", "bad"):
        xlo = min(0.965, np.nanmin(combined) - 0.002)
        xhi = max(1.002, np.nanmax(combined) + 0.002)
    else:
        span = np.nanmax(combined) - np.nanmin(combined)
        pad = max(0.03, 0.08 * span if span > 0 else 0.05)
        xlo = np.nanmin(combined) - pad
        xhi = np.nanmax(combined) + pad

    kernel_ranges[kind] = (xlo, xhi)

fig, axs = plt.subplots(
    nrows=2,
    ncols=3,
    figsize=(13.5, 5.8),
    sharex=False,
    sharey=False,
)

mode_colors = ["tab:blue", "tab:orange", "tab:green"]

for row, (method, _) in enumerate(methods):
    for col, kind in enumerate(kernels):
        ax = axs[row, col]
        vals = np.asarray(results[(kind, method)], dtype=float)  # shape (n_trials, n_show)

        xlo, xhi = kernel_ranges[kind]
        bins = np.linspace(xlo, xhi, 60)

        # plot mode-wise empirical distributions
        for m in range(min(n_show, vals.shape[1])):
            vm = vals[:, m]
            vm = vm[np.isfinite(vm)]
            if vm.size == 0:
                continue

            ax.hist(
                vm,
                bins=bins,
                density=True,
                histtype="step",
                lw=1.8,
                color=mode_colors[m % len(mode_colors)],
                label=f"Mode {m + 1}" if (row == 0 and col == 0) else None,
            )

        # true Koopman eigenvalue lines, only if visible in this panel
        for j, ev in enumerate(true_eigs[:n_show]):
            if xlo <= ev <= xhi:
                ax.axvline(
                    float(ev),
                    color="black",
                    ls="--",
                    lw=1.0,
                    alpha=0.8,
                    label="True Koopman eigenvalue" if (row == 0 and col == 0 and j == 0) else None,
                )

        ax.set_title(f"{method} / {kind}", fontsize=12)
        ax.set_xlim(xlo, xhi)
        ax.grid(alpha=0.20)

        if col == 0:
            ax.set_ylabel("Density")
        if row == len(methods) - 1:
            ax.set_xlabel("Estimated eigenvalue")

legend_handles = [
    Line2D([0], [0], color="tab:blue", lw=1.8, label="Mode 1"),
    Line2D([0], [0], color="tab:orange", lw=1.8, label="Mode 2"),
    Line2D([0], [0], color="tab:green", lw=1.8, label="Mode 3"),
    Line2D([0], [0], color="black", lw=1.0, ls="--", label="True Koopman eigenvalue"),
]

fig.legend(
    handles=legend_handles,
    loc="upper center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, 1.03),
)

fig.suptitle(
    "OU spectrum under good, bad, and ugly kernels",
    fontsize=14,
    y=1.08,
)

plt.tight_layout()
plt.show()


fig.suptitle(
    "OU spectrum under good, bad, and ugly kernels",
    fontsize=14,
    y=1.08,
)

plt.tight_layout()
plt.show()


### Full final version

In [ ]:
# Spectral rates paper fig. 1-style eigenfunction comparison:
# - 3 columns = good / bad / ugly kernels
# - overlay PCR (blue) and RRR (orange)
# - show mean estimated eigenfunction across trials for the first few modes

from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from tqdm import tqdm

# -----------------------------
# Parameters
# -----------------------------
kernels = ["good", "bad", "ugly"]
methods = [("PCR", False), ("RRR", True)]

n_train = 2000
subsample = 100
n_steps = n_train * subsample
n_trials = 50  # increase toward paper settings for smoother curves
n_show = 3  # number of leading eigenfunctions to plot
r = n_show
M = 10  # kernel truncation level
dt = 1e-4
lag = dt * subsample  # now consistent with how `data` was actually generated
n_components = 5
true_eigs = compute_ou_eig(gamma, lag, n_show)
alpha = 1e-10
x_plot = x.reshape(-1)


# -----------------------------
# Fit model and return eigenvalues
# -----------------------------
def fit_and_estimate_values(reduced_rank, x, density, random_state, kind, r, M, n_show):
    out = np.full(n_show, np.nan, dtype=float)

    try:
        data = simulate_ou(
            n_steps=n_steps,
            gamma=gamma,
            dt=dt,
            random_state=random_state,   # IMPORTANT: independent trials
            x0=0.0,
        )
        data = data.iloc[::subsample]
        data = data[:n_train]

        model = KernelRidge(
            n_components=n_show,
            reduced_rank=reduced_rank,
            kernel=hermite_kernel(kind, r, M),
            alpha=alpha,
            random_state=random_state,
        )

        model.fit(data)
        values, functions = model.eig(eval_right_on=x)

        values = np.asarray(values)
        values = np.real_if_close(values)

        if values.ndim == 0:
            values = np.array([values])

        # keep only approximately real eigenvalues
        real_mask = np.abs(np.imag(values)) < alpha
        values = np.real(values[real_mask])

        # keep only finite values
        values = values[np.isfinite(values)]

        # exclude values slightly above 1 from numerical noise
        values = values[values <= 1.0 + alpha]

        # sort by value, largest first
        values = np.sort(values)[::-1]

        k = min(n_show, values.size)
        out[:k] = values[:k]

    except Exception:
        pass

    return out


# -----------------------------
# Collect spectra
# -----------------------------
results = defaultdict(list)

for kind in kernels:
    for method, reduced_rank in methods:
        for trial in tqdm(range(n_trials), desc=f"{kind}-{method}"):
            vals = fit_and_estimate_values(
                reduced_rank=reduced_rank,
                x=x,
                density=density,
                random_state=trial,
                kind=kind,
                r=r,
                M=M,
                n_show=n_show,
            )
            results[(kind, method)].append(np.asarray(vals, dtype=float))


for key, arr_list in results.items():
    arr = np.stack(arr_list, axis=0)
    valid_cols = np.any(np.isfinite(arr), axis=0)
    mean_vals = np.full(arr.shape[1], np.nan)
    mean_vals[valid_cols] = np.nanmean(arr[:, valid_cols], axis=0)
    print(key, mean_vals)

# -----------------------------
# Shared x-range per kernel
# -----------------------------
kernel_ranges = {}
for kind in kernels:
    combined = []
    for method, _ in methods:
        arr = results[(kind, method)]
        combined.append(arr[np.isfinite(arr)])
    combined = np.concatenate(combined) if len(combined) else np.array([])

    if combined.size == 0:
        kernel_ranges[kind] = (0.0, 1.0)
        continue

    if kind in ("good", "bad"):
        xlo = min(np.min(true_eigs) - 0.01, np.nanmin(combined) - 0.005)
        xhi = max(np.max(true_eigs) + 0.005, np.nanmax(combined) + 0.005)
    else:
        span = np.nanmax(combined) - np.nanmin(combined)
        pad = max(0.03, 0.08 * span if span > 0 else 0.05)
        xlo = np.nanmin(combined) - pad
        xhi = np.nanmax(combined) + pad

    kernel_ranges[kind] = (xlo, xhi)

# -----------------------------
# Plot
# -----------------------------

plt.style.use("default")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "axes.edgecolor": "0.15",
    "axes.linewidth": 0.8,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "font.size": 10,
})

# keep only first 3 leading eigenvalues for Fig. 1
n_plot = 3
true_plot = np.asarray(true_eigs[:n_plot], dtype=float)

# compute tighter x-ranges per kernel, shared across PCR/RRR in each column
kernel_ranges = {}
for kind in kernels:
    combined = []
    for method, _ in methods:
        arr = np.asarray(results[(kind, method)], dtype=float)[:, :n_plot]
        combined.append(arr[np.isfinite(arr)])
    combined = np.concatenate(combined) if len(combined) else np.array([])

    if combined.size == 0:
        kernel_ranges[kind] = (0.0, 1.05)
        continue

    if kind == "good":
        xlo = min(np.min(true_plot) - 0.01, np.nanmin(combined) - 0.005)
        xhi = max(np.max(true_plot) + 0.003, np.nanmax(combined) + 0.003)
    elif kind == "bad":
        xlo = min(np.min(true_plot) - 0.03, np.nanmin(combined) - 0.01)
        xhi = max(np.max(true_plot) + 0.01, np.nanmax(combined) + 0.01)
    else:  # ugly
        span = np.nanmax(combined) - np.nanmin(combined)
        pad = max(0.04, 0.08 * span if span > 0 else 0.06)
        xlo = np.nanmin(combined) - pad
        xhi = np.nanmax(combined) + pad

    kernel_ranges[kind] = (xlo, xhi)

fig, axs = plt.subplots(
    nrows=2,
    ncols=3,
    figsize=(12.6, 5.4),
    sharex=False,
    sharey=False,
)

mode_colors = ["#4C78A8", "#F58518", "#54A24B"]

for row, (method, _) in enumerate(methods):
    for col, kind in enumerate(kernels):
        ax = axs[row, col]
        vals = np.asarray(results[(kind, method)], dtype=float)[:, :n_plot]

        xlo, xhi = kernel_ranges[kind]
        bins = np.linspace(xlo, xhi, 70 if kind != "ugly" else 55)

        # distributions of estimated eigenvalues
        for m in range(min(n_plot, vals.shape[1])):
            vm = vals[:, m]
            vm = vm[np.isfinite(vm)]
            if vm.size == 0:
                continue

            # light fill
            ax.hist(
                vm,
                bins=bins,
                density=True,
                histtype="stepfilled",
                color=mode_colors[m],
                alpha=0.16,
                lw=0,
            )
            # clean outline
            ax.hist(
                vm,
                bins=bins,
                density=True,
                histtype="step",
                color=mode_colors[m],
                lw=1.35,
                label=f"Mode {m + 1}" if (row == 0 and col == 0) else None,
            )

        # true Koopman eigenvalues
        for j, ev in enumerate(true_plot):
            if xlo <= ev <= xhi:
                ax.axvline(
                    float(ev),
                    color="black",
                    lw=0.95,
                    ls=(0, (4, 2)),
                    alpha=0.9,
                    zorder=3,
                    label="Koopman eigenvalue" if (row == 0 and col == 0 and j == 0) else None,
                )

        ax.set_title(f"{method} / {kind}", pad=6)
        ax.set_xlim(xlo, xhi)

        # paper-like axes
        ax.grid(True, axis="y", color="0.88", lw=0.6)
        ax.grid(False, axis="x")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        if col == 0:
            ax.set_ylabel("Density")
        if row == len(methods) - 1:
            ax.set_xlabel("Estimated eigenvalue")

# compact legend
legend_handles = [
    Line2D([0], [0], color=mode_colors[0], lw=1.5, label="Mode 1"),
    Line2D([0], [0], color=mode_colors[1], lw=1.5, label="Mode 2"),
    Line2D([0], [0], color=mode_colors[2], lw=1.5, label="Mode 3"),
    Line2D([0], [0], color="black", lw=0.95, ls=(0, (4, 2)), label="Koopman eigenvalue"),
]

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.995),
    ncol=4,
    frameon=False,
    handlelength=2.2,
    columnspacing=1.6,
)

fig.subplots_adjust(
    top=0.84,
    left=0.07,
    right=0.995,
    bottom=0.12,
    wspace=0.26,
    hspace=0.34,
)

plt.show()


good-PCR:   0%|          | 0/50 [00:00<?, ?it/s]

good-RRR:   4%|▍         | 2/50 [02:11<52:40, 65.84s/it]/home/scj/.local/lib/python3.14/site-packages/kooplearn/_utils.py:77: UserWarning: Warning: Discarded 1 dimensions of the 3 requested due to numerical instability. Consider decreasing k_max. The largest discarded value is: 1.560e-07.
  warn(f"Warning: Discarded {k_max - np.sum(valid)} dimensions of the {k_max} " \
good-RRR:   8%|▊         | 4/50 [04:15<48:30, 63.27s/it]/home/scj/.local/lib/python3.14/site-packages/kooplearn/_utils.py:77: UserWarning: Warning: Discarded 1 dimensions of the 3 requested due to numerical instability. Consider decreasing k_max. The largest discarded value is: 1.567e-07.
  warn(f"Warning: Discarded {k_max - np.sum(valid)} dimensions of the {k_max} " \
good-RRR:  90%|█████████ | 45/50 [5:24:25<05:27, 65.53s/it]     /home/scj/.local/lib/python3.14/site-packages/kooplearn/_utils.py:77: UserWarning: Warning: Discarded 1 dimensions of the 3 requested due to numerical instability. Consider decreasing k_max. T